# 🏠 Smart Home — Module IA V2 (PC Maintenance)

**Artefact 1/4** — Google Colab · réseau de neurones & apprentissage des capteurs.
**Maison :** plain-pied, 8 pièces connectées (cf. plan du projet).

## 🎯 Objectif
Collecter, enregistrer et apprendre 4 grandeurs par pièce — **température, présence, lumière, état de la fenêtre** — pour prédire des déclencheurs d'automatisation (éclairage, chauffage anticipé, routine café, alertes).

## 🔌 Contrat MQTT (V2)
| Topic | Payload | Exemple |
|---|---|---|
| `smarthome/<piece>/temperature` | nombre (°C) | `21.4` |
| `smarthome/<piece>/presence` | 0 / 1 (PIR) | `1` |
| `smarthome/<piece>/light` | 0 / 1 | `1` |
| `smarthome/<piece>/window` | 0 fermée / 1 ouverte | `0` |
| `smarthome/predictions/<piece>` | JSON publié par CE notebook | `{"activity_prob":0.83,"temp_next":21.9}` |

`<piece>` ∈ { `salon`, `cuisine`, `sdb`, `chparents`, `chenf1`, `chenf2`, `garage`, `terrasse` }.

## 🗂️ Versions
| Version | Date | Auteur | Modifications |
|---|---|---|---|
| V1.0 | 2026-06-09 | Équipe + Claude | Création (DHT22, PIR, volets, gsensor — 4 pièces). |
| **V2.0** | 2026-06-09 | Équipe + Claude | Refonte sur le **plan réel 8 pièces** ; capteurs = **température / présence / lumière / fenêtre** (humidité & volets retirés) ; connexion HiveMQ pré-remplie ; générateur synthétique et modèles adaptés. |


## 1. Dépendances

In [ ]:
!pip install -q "paho-mqtt==1.6.1" joblib
import os, json, ssl, time, threading, warnings
from datetime import datetime
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
print("✅ Librairies prêtes")

## 2. Configuration — HiveMQ pré-rempli

In [ ]:
# ----------- HiveMQ Cloud (pré-rempli) -----------
HIVEMQ_HOST = "VOTRE-CLUSTER.s1.eu.hivemq.cloud"
HIVEMQ_PORT = 8883            # TLS (MQTT)
HIVEMQ_USER = "VOTRE_USER"
HIVEMQ_PASS = "VOTRE_MOT_DE_PASSE"

# ----------- Plan : 8 pièces -----------
PIECES   = ["salon","cuisine","sdb","chparents","chenf1","chenf2","garage","terrasse"]
CAPTEURS = ["temperature","presence","light","window"]
BASE_TOPIC = "smarthome"

RESAMPLE = "5min"; LOOKBACK = 12
DATA_DIR="/content/smarthome_data"; MODELS_DIR="/content/smarthome_models"
RAW_CSV=os.path.join(DATA_DIR,"capteurs_raw.csv")
os.makedirs(DATA_DIR,exist_ok=True); os.makedirs(MODELS_DIR,exist_ok=True)
print("✅ Config OK —", len(PIECES), "pièces,", CAPTEURS)

## 3. Persistance Google Drive *(optionnel)*

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    DATA_DIR="/content/drive/MyDrive/SmartHome/data"; MODELS_DIR="/content/drive/MyDrive/SmartHome/models"
    RAW_CSV=os.path.join(DATA_DIR,"capteurs_raw.csv")
    os.makedirs(DATA_DIR,exist_ok=True); os.makedirs(MODELS_DIR,exist_ok=True)
    print("✅ Drive monté")
else:
    print("ℹ️ Drive désactivé (mets USE_DRIVE=True pour persister).")

## 4. Collecte MQTT (HiveMQ TLS)
S'abonne à `smarthome/#` et enregistre au format `timestamp, piece, capteur, valeur`.

In [ ]:
import paho.mqtt.client as mqtt
_buf=[]; _lock=threading.Lock()
def _parse(topic,payload):
    p=topic.split("/"); ts=datetime.utcnow().isoformat(); rows=[]
    try:
        if len(p)==3 and p[0]==BASE_TOPIC and p[1] in PIECES and p[2] in CAPTEURS:
            try: v=float(payload)
            except: v=float(json.loads(payload).get("value"))
            rows.append((ts,p[1],p[2],v))
    except Exception as e: print("⚠️",topic,e)
    return rows
def _on_conn(c,u,f,rc): print("🔗 Connecté rc=%s"%rc); c.subscribe(BASE_TOPIC+"/#",qos=1)
def _on_msg(c,u,m):
    r=_parse(m.topic,m.payload.decode("utf-8","ignore"))
    if r:
        with _lock: _buf.extend(r)
def collecter_mqtt(duree_s=60):
    global _buf
    with _lock: _buf=[]
    c=mqtt.Client(client_id="colab-maint-v2",protocol=mqtt.MQTTv311)
    c.username_pw_set(HIVEMQ_USER,HIVEMQ_PASS); c.tls_set(tls_version=ssl.PROTOCOL_TLS_CLIENT)
    c.on_connect=_on_conn; c.on_message=_on_msg
    c.connect(HIVEMQ_HOST,HIVEMQ_PORT,60); c.loop_start()
    print("⏳ collecte %ss..."%duree_s); time.sleep(duree_s); c.loop_stop(); c.disconnect()
    with _lock: rows=list(_buf)
    if not rows: print("⚠️ aucun message reçu"); return pd.DataFrame(columns=["timestamp","piece","capteur","valeur"])
    df=pd.DataFrame(rows,columns=["timestamp","piece","capteur","valeur"])
    df.to_csv(RAW_CSV,mode="a",header=not os.path.exists(RAW_CSV),index=False)
    print("✅ %d relevés -> %s"%(len(df),RAW_CSV)); return df
print("✅ Collecteur prêt — collecter_mqtt(120) quand les capteurs émettent.")

## 5. Générateur synthétique *(test sans capteurs)*
Reproduit température, **présence**, **lumière** et **fenêtre** sur 14 jours avec des routines réalistes.

In [ ]:
PROFILS = {
 "salon":[(18,23,0.85)], "cuisine":[(7,9,0.8),(12,13,0.6),(19,21,0.8)],
 "sdb":[(7,8,0.7),(22,23,0.6)], "chparents":[(22,24,0.7),(0,7,0.5)],
 "chenf1":[(21,23,0.7),(6,8,0.5)], "chenf2":[(21,23,0.7),(6,8,0.5)],
 "garage":[(8,9,0.5),(18,19,0.5)], "terrasse":[(12,16,0.5)] }

def generer_synthetique(jours=14, seed=42):
    rng=np.random.default_rng(seed)
    idx=pd.date_range(end=datetime.utcnow(), periods=int(jours*24*60/5), freq="5min"); rows=[]
    for t in idx:
        h=t.hour+t.minute/60; weekend=t.dayofweek>=5; shift=1.5 if weekend else 0.0
        for pc in PIECES:
            base=20+2.5*np.sin((h-15)/24*2*np.pi)+(2 if pc=="sdb" else 0)-(3 if pc in("garage","terrasse") else 0)
            temp=base+rng.normal(0,0.3)
            p=0.03
            for (a,b,inten) in PROFILS.get(pc,[]):
                if (a+shift)<=h<(b+shift): p=inten
            presence=1 if rng.random()<p else 0
            # lumière : présence + obscurité (ou aléa)
            dark = (h<7.5 or h>=19)
            light=1 if (presence and dark) or (rng.random()<0.03) else 0
            # fenêtre : ouverte en journée pour pièces de vie, fermée la nuit
            jour = 10<=h<18
            popen = 0.6 if (jour and pc in("salon","chparents","chenf1","chenf2")) else (0.2 if jour and pc=="cuisine" else 0.03)
            window=1 if rng.random()<popen else 0
            rows += [(t.isoformat(),pc,"temperature",round(float(temp),2)),
                     (t.isoformat(),pc,"presence",int(presence)),
                     (t.isoformat(),pc,"light",int(light)),
                     (t.isoformat(),pc,"window",int(window))]
    return pd.DataFrame(rows,columns=["timestamp","piece","capteur","valeur"])

df_synth=generer_synthetique(); df_synth.to_csv(RAW_CSV,index=False)
print("✅ %d relevés synthétiques générés"%len(df_synth)); df_synth.head()

## 6. Chargement & features
Format long → large + features temporelles cycliques (heure / jour / week-end).

In [ ]:
def charger_long(p=RAW_CSV):
    d=pd.read_csv(p); d["timestamp"]=pd.to_datetime(d["timestamp"]); return d
def to_wide(dl):
    w=dl.pivot_table(index="timestamp",columns=["piece","capteur"],values="valeur",aggfunc="mean")
    w.columns=["%s_%s"%(a,b) for a,b in w.columns]
    return w.sort_index().resample(RESAMPLE).mean().interpolate().ffill().bfill()
def feats_temps(w):
    w=w.copy(); h=w.index.hour+w.index.minute/60; d=w.index.dayofweek
    w["hour_sin"]=np.sin(2*np.pi*h/24); w["hour_cos"]=np.cos(2*np.pi*h/24)
    w["dow_sin"]=np.sin(2*np.pi*d/7);  w["dow_cos"]=np.cos(2*np.pi*d/7)
    w["is_weekend"]=(d>=5).astype(int); return w
wide=feats_temps(to_wide(charger_long()))
print("✅ Tableau large:",wide.shape); wide.head()

## 7. Analyse exploratoire

In [ ]:
tcols=[c for c in wide.columns if c.endswith("_temperature")]
plt.figure(figsize=(12,4))
for c in tcols: plt.plot(wide.index,wide[c],label=c.replace("_temperature",""),alpha=.8)
plt.title("Température par pièce"); plt.ylabel("°C"); plt.legend(ncol=4,fontsize=8); plt.tight_layout(); plt.show()

pcols=[c for c in wide.columns if c.endswith("_presence")]
pr=wide.copy(); pr["h"]=pr.index.hour
plt.figure(figsize=(12,4))
for c in pcols: plt.plot(pr.groupby("h")[c].mean(),marker="o",label=c.replace("_presence",""))
plt.title("Probabilité de présence par heure"); plt.xlabel("Heure"); plt.legend(ncol=4,fontsize=8); plt.tight_layout(); plt.show()

## 8. Modèle A — Présence par pièce *(apprentissage séparé)*
Un réseau de neurones par pièce prédit la **probabilité de présence** (features : temps + température + fenêtre).

In [ ]:
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib
FEATS=["hour_sin","hour_cos","dow_sin","dow_cos","is_weekend"]
def build(n):
    m=keras.Sequential([keras.layers.Input((n,)),keras.layers.Dense(32,activation="relu"),
        keras.layers.Dropout(.2),keras.layers.Dense(16,activation="relu"),keras.layers.Dense(1,activation="sigmoid")])
    m.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy",keras.metrics.AUC(name="auc")]); return m
modeles={}
for pc in PIECES:
    tgt="%s_presence"%pc
    if tgt not in wide.columns: continue
    f=FEATS.copy()
    for e in ["%s_temperature"%pc,"%s_window"%pc]:
        if e in wide.columns: f.append(e)
    X=wide[f].values; y=(wide[tgt]>0.5).astype(int).values
    sc=StandardScaler().fit(X); Xs=sc.transform(X)
    Xtr,Xte,ytr,yte=train_test_split(Xs,y,test_size=.2,shuffle=False)
    m=build(len(f)); es=keras.callbacks.EarlyStopping(patience=8,restore_best_weights=True)
    hh=m.fit(Xtr,ytr,validation_data=(Xte,yte),epochs=60,batch_size=32,verbose=0,callbacks=[es])
    _,acc,auc=m.evaluate(Xte,yte,verbose=0)
    print("🏠 %-9s acc=%.3f AUC=%.3f"%(pc,acc,auc))
    modeles[pc]={"model":m,"scaler":sc,"features":f,"history":hh.history}
print("\n✅ %d modèles de présence entraînés."%len(modeles))

## 9. Modèle B — LSTM prévision de température (salon)
Pour le chauffage anticipé.

In [ ]:
def seqs(a,lb):
    X,y=[],[]
    for i in range(len(a)-lb): X.append(a[i:i+lb]); y.append(a[i+lb])
    return np.array(X),np.array(y)
PIECE_LSTM="salon"; serie=wide["%s_temperature"%PIECE_LSTM].values.reshape(-1,1)
sct=StandardScaler().fit(serie); s=sct.transform(serie).flatten()
X,yv=seqs(s,LOOKBACK); X=X.reshape((X.shape[0],LOOKBACK,1)); n=int(len(X)*.8)
lstm=keras.Sequential([keras.layers.Input((LOOKBACK,1)),keras.layers.LSTM(32),
    keras.layers.Dense(16,activation="relu"),keras.layers.Dense(1)])
lstm.compile(optimizer="adam",loss="mse",metrics=["mae"])
es=keras.callbacks.EarlyStopping(patience=8,restore_best_weights=True)
lstm.fit(X[:n],yv[:n],validation_data=(X[n:],yv[n:]),epochs=40,batch_size=32,verbose=0,callbacks=[es])
mse,mae=lstm.evaluate(X[n:],yv[n:],verbose=0); print("🌡️ LSTM %s MSE=%.4f MAE=%.4f"%(PIECE_LSTM,mse,mae))

## 10. Évaluation

In [ ]:
p0=list(modeles.keys())[0]; h=modeles[p0]["history"]
plt.figure(figsize=(12,4))
plt.subplot(1,2,1); plt.plot(h["loss"],label="train"); plt.plot(h["val_loss"],label="val"); plt.title("Présence %s — loss"%p0); plt.legend()
plt.subplot(1,2,2); plt.plot(h["accuracy"],label="train"); plt.plot(h["val_accuracy"],label="val"); plt.title("Accuracy"); plt.legend()
plt.tight_layout(); plt.show()
pred=lstm.predict(X[n:],verbose=0).flatten()
pr=sct.inverse_transform(pred.reshape(-1,1)).flatten(); tr=sct.inverse_transform(yv[n:].reshape(-1,1)).flatten()
plt.figure(figsize=(12,4)); plt.plot(tr[:200],label="réel"); plt.plot(pr[:200],label="prédit",alpha=.8)
plt.title("LSTM salon — prévision température"); plt.ylabel("°C"); plt.legend(); plt.tight_layout(); plt.show()

## 11. Inférence, publication MQTT (→ HUD / Node-RED) & export

In [ ]:
def predire():
    now=datetime.utcnow(); h=now.hour+now.minute/60; d=now.weekday()
    base={"hour_sin":np.sin(2*np.pi*h/24),"hour_cos":np.cos(2*np.pi*h/24),
          "dow_sin":np.sin(2*np.pi*d/7),"dow_cos":np.cos(2*np.pi*d/7),"is_weekend":1 if d>=5 else 0}
    out={}
    for pc,pk in modeles.items():
        v=[base[f] if f in base else float(wide[f].iloc[-1]) for f in pk["features"]]
        x=pk["scaler"].transform(np.array(v).reshape(1,-1))
        out[pc]={"activity_prob":round(float(pk["model"].predict(x,verbose=0)[0,0]),3)}
    ls=s[-LOOKBACK:].reshape(1,LOOKBACK,1)
    out.setdefault(PIECE_LSTM,{})["temp_next"]=round(float(sct.inverse_transform(lstm.predict(ls,verbose=0))[0,0]),2)
    return out
pred=predire(); print("🔮", json.dumps(pred,indent=2))

def publier(pred):
    c=mqtt.Client(client_id="colab-pred-v2",protocol=mqtt.MQTTv311)
    c.username_pw_set(HIVEMQ_USER,HIVEMQ_PASS); c.tls_set(tls_version=ssl.PROTOCOL_TLS_CLIENT)
    c.connect(HIVEMQ_HOST,HIVEMQ_PORT,30); c.loop_start()
    for pc,pl in pred.items():
        c.publish("%s/predictions/%s"%(BASE_TOPIC,pc),json.dumps(pl),qos=1,retain=True); print("📤",pc,pl)
    time.sleep(1); c.loop_stop(); c.disconnect()
# publier(pred)   # décommente pour envoyer vers le HUD / Node-RED

for pc,pk in modeles.items():
    pk["model"].save(os.path.join(MODELS_DIR,"presence_%s.keras"%pc))
    joblib.dump(pk["scaler"],os.path.join(MODELS_DIR,"scaler_%s.pkl"%pc))
lstm.save(os.path.join(MODELS_DIR,"lstm_temp_%s.keras"%PIECE_LSTM))
print("✅ Modèles exportés ->",MODELS_DIR)

## 12. 🗂️ Traçabilité — détail

| Version | Date | Auteur | Détail |
|---|---|---|---|
| V1.0 | 2026-06-09 | Équipe + Claude | Pipeline initial (4 pièces ; DHT22/humidité, PIR, volets, gsensor). |
| **V2.0** | 2026-06-09 | Équipe + Claude | Alignement sur le **plan réel 8 pièces** (salon, cuisine, sdb, chparents, chenf1, chenf2, garage, terrasse). Capteurs ramenés à **température / présence / lumière / fenêtre**. Connexion **HiveMQ pré-remplie**. Générateur synthétique réécrit (présence + lumière liée à l'obscurité + fenêtre liée au jour). Modèle A = présence par pièce (features temps + température + fenêtre). LSTM température conservé. Publication des prédictions inchangée. |

### TODO V2.1
- [ ] Brancher `collecter_mqtt()` sur les vrais capteurs.
- [ ] Ajouter un modèle « fenêtre ouverte probable » (sécurité / aération).
- [ ] Scénario voiture-approche (nécessite un capteur de proximité au garage).
